# Smoking Detection - Train YOLO26n on Google Colab
Uses free GPU (T4) to train YOLO26n on the Roboflow smoking dataset.

**Runtime → Change runtime type → GPU (T4)**

In [ ]:
# Step 1: Install ultralytics
!pip install ultralytics roboflow -q

In [ ]:
# Step 2: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Step 3: Download dataset from Roboflow (YOLO26 format)
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")  # Get from https://app.roboflow.com/settings/api
project = rf.workspace("richie-lab").project("smoking-tasfx")
version = project.version(4)
dataset = version.download("yolov8")  # yolov8 format works with YOLO26

print(f"Dataset location: {dataset.location}")

In [ ]:
# Step 4: Train YOLO26n
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # auto-downloads pretrained weights

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,        # GPU
    workers=2,
    project="smoking-detection",
    name="yolo26n",
)

In [ ]:
# Step 5: Check results
from IPython.display import Image, display

# Training curves
display(Image(filename="smoking-detection/yolo26n/results.png", width=800))

# Confusion matrix
display(Image(filename="smoking-detection/yolo26n/confusion_matrix.png", width=600))

In [ ]:
# Step 6: Test on validation images
best_model = YOLO("smoking-detection/yolo26n/weights/best.pt")
metrics = best_model.val()
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Step 7: Export to NCNN (optimized for Raspberry Pi ARM CPU)
best_model.export(format="ncnn")
print("NCNN model exported!")

In [ ]:
# Step 8: Download weights
# Option A: Download best.pt (PyTorch)
from google.colab import files
files.download("smoking-detection/yolo26n/weights/best.pt")

# Option B: Download NCNN model (faster on Pi)
!zip -r ncnn_model.zip smoking-detection/yolo26n/weights/best_ncnn_model/
files.download("ncnn_model.zip")